<a href="https://colab.research.google.com/github/ritvik-123/114-Assignments-OS/blob/main/Final_LogReg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# --- Imports -----------------------------------------------------------
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD         # dimensionality reduction (SVD works with dense/sparse)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from sklearn.metrics import top_k_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

import torch

In [4]:
# --- Config --------------------------------------------------------------
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"   # your labeled sentence-level file

TARGET_LABELS = [                                # the 4 label columns we're predicting
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized",
]
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"  # 1024-dim embedding model
N_SVD_COMPONENTS = 50                            # compress 384-dim embeddings down further before the classifier
N_FOLDS = 5                                      # number of GroupKFold splits (you could also use LeaveOneGroupOut)
RANDOM_STATE = 42                                # fixed seed so results are reproducible
THRESHOLD = 0.5                                  # probability cutoff for turning a score into a 0/1 prediction

%pwd
%cd /content/drive/MyDrive/EMP/Notebook

/content/drive/MyDrive/EMP/Notebook


In [5]:
# --- Load -----------------------------------------------------------------
df = pd.read_csv(CSV_PATH + "Generated_Sentences_1.csv")

df_1 = pd.read_csv(CSV_PATH + "Module 1 Sentences Gemini.csv")

df_1 = df_1[['sentence','ideological','institutionalized','interpersonal','internalized','primary_leaning']]
df_1 = df_1[df_1['primary_leaning'] != 'none']
df_1 = df_1.reset_index(drop=True)
df_1 = df_1.rename(columns={'sentence':'Sentence'})
df_1 = df_1.rename(columns={'primary_leaning':'Label'})
df_1["data_source"] = "real world"

df = pd.concat([df, df_1], ignore_index=True)

# df = df[df['data_source'] != 'extra_institutionalized']

# Clean column names just in case there are spaces
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())

# Clean sentences
df["Sentence"] = df["Sentence"].fillna("").astype(str).str.strip()
df = df[df["Sentence"] != ""].reset_index(drop=True)

# Clean Label column
df["Label"] = df["Label"].fillna("").astype(str).str.strip().str.lower()

# Create the 4 target label columns from the single Label column
for label in TARGET_LABELS:
    df[label] = (df["Label"] == label).astype(int)

# Optional: create source_id if your new file does not have one
if "source_id" not in df.columns:
    df["source_id"] = df.index

print("Rows after cleaning:", len(df))
print("Unique groups (source_id):", df["source_id"].nunique())
print(df[TARGET_LABELS].sum())

Columns: ['Sentence', 'Label', 'data_source', 'ideological', 'institutionalized', 'interpersonal', 'internalized']
Rows after cleaning: 810
Unique groups (source_id): 810
ideological          203
institutionalized    256
interpersonal        195
internalized         156
dtype: int64


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [7]:
# --- Embed ------------------------------------------------------------------
device = "cuda"   # change to "cpu" if you're not on a GPU runtime

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)   # loads the pretrained embedding model

sentences = df["Sentence"].tolist()                                   # plain list of strings to embed

X_full = embedder.encode(
    sentences,                     # the sentences to embed
    batch_size=32,                 # how many sentences to embed per forward pass
    normalize_embeddings=True,     # L2-normalize each embedding (helps cosine-similarity-style comparisons)
    show_progress_bar=True,        # display a progress bar since this can take a minute
    convert_to_numpy=True,         # return a numpy array instead of torch tensors
)

print("Embedding matrix shape:", X_full.shape)   # expect (num_sentences, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Embedding matrix shape: (810, 1024)


In [8]:
# ------------------------------------------------------------
# Train/test split BEFORE SVD + Logistic Regression
# ------------------------------------------------------------

y_full = df["Label"].values

X_train_embed, X_test_embed, y_train, y_test, df_train, df_test = train_test_split(
    X_full,
    y_full,
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print("Train shape:", X_train_embed.shape)
print("Test shape:", X_test_embed.shape)

Train shape: (648, 1024)
Test shape: (162, 1024)


In [9]:
# ------------------------------------------------------------
# Sample weights for TRAINING rows only
# ------------------------------------------------------------

sample_weight = np.ones(len(df_train))

sample_weight[df_train["data_source"] == "base_synthetic"] = 1.0
sample_weight[df_train["data_source"] == "contrastive_synthetic"] = 1.0
sample_weight[df_train["data_source"] == "contrastive_synthetic_v2"] = 0.8
sample_weight[df_train["data_source"] == "extra_institutionalized"] = 1.0

In [10]:
# ------------------------------------------------------------
# Fit SVD on train only, then transform train and test
# ------------------------------------------------------------

final_svd = TruncatedSVD(
    n_components=N_SVD_COMPONENTS,
    random_state=RANDOM_STATE
)

X_train = final_svd.fit_transform(X_train_embed)
X_test = final_svd.transform(X_test_embed)

In [11]:
# ------------------------------------------------------------
# Train Logistic Regression
# ------------------------------------------------------------

final_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        C=0.5,
        max_iter=3000,
        solver="lbfgs",
        class_weight=None,
        random_state=RANDOM_STATE
    ))
])

final_clf.fit(
    X_train,
    y_train,
    logreg__sample_weight=sample_weight
)

print("Model trained on training split.")
print("Classes:", final_clf.named_steps["logreg"].classes_)

Model trained on training split.
Classes: ['ideological' 'institutionalized' 'internalized' 'interpersonal']


In [12]:
# ------------------------------------------------------------
# Evaluate on test set
# ------------------------------------------------------------

y_pred = final_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=final_clf.named_steps["logreg"].classes_))

Accuracy: 0.8518518518518519
Macro F1: 0.8448306806515762

Classification report:
                   precision    recall  f1-score   support

      ideological       0.75      0.73      0.74        41
institutionalized       0.98      0.92      0.95        51
     internalized       0.75      0.87      0.81        31
    interpersonal       0.89      0.87      0.88        39

         accuracy                           0.85       162
        macro avg       0.84      0.85      0.84       162
     weighted avg       0.86      0.85      0.85       162


Confusion matrix:
[[30  1  7  3]
 [ 4 47  0  0]
 [ 3  0 27  1]
 [ 3  0  2 34]]


In [13]:
# ------------------------------------------------------------
# Evaluate Top-2 accuracy
# ------------------------------------------------------------

proba = final_clf.predict_proba(X_test)

classes = final_clf.named_steps["logreg"].classes_

top2_acc = top_k_accuracy_score(
    y_test,
    proba,
    k=2,
    labels=classes
)

print("Top-2 Accuracy:", top2_acc)

Top-2 Accuracy: 0.9753086419753086


In [14]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# ------------------------------------------------------------
# Hyperparameter tuning on TRAINING DATA ONLY
# ------------------------------------------------------------

tuning_pipe = Pipeline([
    ("svd", TruncatedSVD(random_state=RANDOM_STATE)),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        random_state=RANDOM_STATE
    ))
])

param_grid = {
    "svd__n_components": [30, 50, 75, 100, 150, 200],
    "logreg__C": [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
    "logreg__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

grid = GridSearchCV(
    estimator=tuning_pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(
    X_train_embed,
    y_train,
    logreg__sample_weight=sample_weight
)

print("Best CV Macro F1:", grid.best_score_)
print("Best params:", grid.best_params_)

Fitting 5 folds for each of 84 candidates, totalling 420 fits
Best CV Macro F1: 0.8735192565498204
Best params: {'logreg__C': 0.05, 'logreg__class_weight': None, 'svd__n_components': 150}


In [15]:
# ------------------------------------------------------------
# Evaluate best tuned model on untouched test set
# ------------------------------------------------------------

best_clf = grid.best_estimator_

y_pred_best = best_clf.predict(X_test_embed)
proba_best = best_clf.predict_proba(X_test_embed)

classes_best = best_clf.named_steps["logreg"].classes_

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Macro F1:", f1_score(y_test, y_pred_best, average="macro"))

print("Top-2 Accuracy:", top_k_accuracy_score(
    y_test,
    proba_best,
    k=2,
    labels=classes_best
))

print("\nClassification report:")
print(classification_report(y_test, y_pred_best))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_best, labels=classes_best))

Accuracy: 0.8209876543209876
Macro F1: 0.8120765521398433
Top-2 Accuracy: 0.9691358024691358

Classification report:
                   precision    recall  f1-score   support

      ideological       0.71      0.66      0.68        41
institutionalized       0.96      0.92      0.94        51
     internalized       0.69      0.87      0.77        31
    interpersonal       0.89      0.82      0.85        39

         accuracy                           0.82       162
        macro avg       0.81      0.82      0.81       162
     weighted avg       0.83      0.82      0.82       162


Confusion matrix:
[[27  2  9  3]
 [ 2 47  2  0]
 [ 3  0 27  1]
 [ 6  0  1 32]]


In [16]:
# ------------------------------------------------------------
# Logistic Regression without SVD
# ------------------------------------------------------------

no_svd_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        C=0.5,
        max_iter=5000,
        solver="lbfgs",
        class_weight=None,
        random_state=RANDOM_STATE
    ))
])

no_svd_clf.fit(
    X_train_embed,
    y_train,
    logreg__sample_weight=sample_weight
)

y_pred_no_svd = no_svd_clf.predict(X_test_embed)
proba_no_svd = no_svd_clf.predict_proba(X_test_embed)

classes_no_svd = no_svd_clf.named_steps["logreg"].classes_

print("No SVD Accuracy:", accuracy_score(y_test, y_pred_no_svd))
print("No SVD Macro F1:", f1_score(y_test, y_pred_no_svd, average="macro"))
print("No SVD Top-2 Accuracy:", top_k_accuracy_score(
    y_test,
    proba_no_svd,
    k=2,
    labels=classes_no_svd
))

No SVD Accuracy: 0.8271604938271605
No SVD Macro F1: 0.8150408889677165
No SVD Top-2 Accuracy: 0.9753086419753086


In [17]:
from sklearn.calibration import CalibratedClassifierCV

# ------------------------------------------------------------
# Calibrated Logistic Regression
# ------------------------------------------------------------

base_clf = Pipeline([
    ("svd", TruncatedSVD(
        n_components=grid.best_params_["svd__n_components"],
        random_state=RANDOM_STATE
    )),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        C=grid.best_params_["logreg__C"],
        max_iter=5000,
        solver="lbfgs",
        class_weight=grid.best_params_["logreg__class_weight"],
        random_state=RANDOM_STATE
    ))
])

calibrated_clf = CalibratedClassifierCV(
    estimator=base_clf,
    method="sigmoid",
    cv=5
)

calibrated_clf.fit(
    X_train_embed,
    y_train,
    logreg__sample_weight=sample_weight
)

y_pred_cal = calibrated_clf.predict(X_test_embed)
proba_cal = calibrated_clf.predict_proba(X_test_embed)

print("Calibrated Accuracy:", accuracy_score(y_test, y_pred_cal))
print("Calibrated Macro F1:", f1_score(y_test, y_pred_cal, average="macro"))
print("Calibrated Top-2 Accuracy:", top_k_accuracy_score(
    y_test,
    proba_cal,
    k=2,
    labels=calibrated_clf.classes_
))

Calibrated Accuracy: 0.8271604938271605
Calibrated Macro F1: 0.8147696931419736
Calibrated Top-2 Accuracy: 0.9753086419753086


In [18]:
def predict_with_top2(sentence, embedder, clf):
    embedding = embedder.encode(
        [sentence],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    proba = clf.predict_proba(embedding)[0]
    classes = clf.classes_ if hasattr(clf, "classes_") else clf.named_steps["logreg"].classes_

    top2_idx = np.argsort(proba)[-2:][::-1]

    top1_label = classes[top2_idx[0]]
    top2_label = classes[top2_idx[1]]

    top1_prob = proba[top2_idx[0]]
    top2_prob = proba[top2_idx[1]]

    margin = top1_prob - top2_prob

    result = {
        "top1_label": top1_label,
        "top1_prob": float(top1_prob),
        "top2_label": top2_label,
        "top2_prob": float(top2_prob),
        "margin": float(margin)
    }

    return result

In [19]:
result = predict_with_top2(
    "I started believing I was less capable because of how people treated me.",
    embedder,
    best_clf
)

result

{'top1_label': 'interpersonal',
 'top1_prob': 0.5165428015485243,
 'top2_label': 'internalized',
 'top2_prob': 0.4406003820690094,
 'margin': 0.07594241947951486}

In [20]:
result = predict_with_top2(
    "The historic practice of redlining by major banks systematically denied mortgages to minority applicants, trapping generations of families in underfunded neighborhoods.",
    embedder,
    best_clf
)

result

{'top1_label': 'institutionalized',
 'top1_prob': 0.9943352310684986,
 'top2_label': 'internalized',
 'top2_prob': 0.0021094880751017417,
 'margin': 0.9922257429933969}

In [21]:
result = predict_with_top2(
    "The cultural myth of the 'model minority' places immense psychological pressure on Asian Americans while simultaneously being used to mask systemic racism and invalidate the struggles of other marginalized groups.",
    embedder,
    best_clf
)

result

{'top1_label': 'ideological',
 'top1_prob': 0.9426492977522043,
 'top2_label': 'institutionalized',
 'top2_prob': 0.02846571058245376,
 'margin': 0.9141835871697506}

In [29]:
result = predict_with_top2(
    "A talented student from a working-class background who refuses to apply for a prestigious scholarship because they believe people from their neighborhood 'don't belong' at elite universities.",
    embedder,
    best_clf
)

result

{'top1_label': 'interpersonal',
 'top1_prob': 0.41919594741377364,
 'top2_label': 'ideological',
 'top2_prob': 0.317028915235124,
 'margin': 0.10216703217864964}

In [30]:
thresholds = []

proba = final_clf.predict_proba(X_test)
classes = final_clf.named_steps["logreg"].classes_

top2_idx = np.argsort(proba, axis=1)[:, -2:][:, ::-1]

top1 = classes[top2_idx[:, 0]]
top2 = classes[top2_idx[:, 1]]

p1 = proba[np.arange(len(proba)), top2_idx[:, 0]]
p2 = proba[np.arange(len(proba)), top2_idx[:, 1]]

margin = p1 - p2
correct = top1 == y_test

for conf_threshold in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
    for margin_threshold in [0.05, 0.10, 0.15, 0.20]:
        send_to_llm = (p1 < conf_threshold) | (margin < margin_threshold)
        keep_logreg = ~send_to_llm

        if keep_logreg.sum() == 0:
            continue

        thresholds.append({
            "conf_threshold": conf_threshold,
            "margin_threshold": margin_threshold,
            "logreg_kept_percent": keep_logreg.mean(),
            "llm_fallback_percent": send_to_llm.mean(),
            "logreg_accuracy_when_kept": correct[keep_logreg].mean()
        })

threshold_df = pd.DataFrame(thresholds)

threshold_df.sort_values(
    ["logreg_accuracy_when_kept", "logreg_kept_percent"],
    ascending=False
)

,conf_threshold,margin_threshold,logreg_kept_percent,llm_fallback_percent,logreg_accuracy_when_kept
20,0.75,0.05,0.777778,0.222222,0.936508
21,0.75,0.10,0.777778,0.222222,0.936508
22,0.75,0.15,0.777778,0.222222,0.936508
23,0.75,0.20,0.777778,0.222222,0.936508
16,0.70,0.05,0.802469,0.197531,0.915385
17,0.70,0.10,0.802469,0.197531,0.915385
18,0.70,0.15,0.802469,0.197531,0.915385
19,0.70,0.20,0.802469,0.197531,0.915385
12,0.65,0.05,0.833333,0.166667,0.903704
13,0.65,0.10,0.833333,0.166667,0.903704
